L'idée est de tester si la simple représentation sous forme de graphe avec une fonction de coût adaptée fournit les mêmes résultats pour le transport, que ceux obtenus en utilisant les *embeddings*. Cela permet de valider ou pas la pertinence d'une telle approche.

## Imports

In [4]:
import torch
import pandas as pd
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from collections import deque, defaultdict
from tqdm import tqdm
from data_utils import *
from OT_utils import *
from information_content import *

## Chargement des données

In [5]:
hp_ids = []
parents_list = []

with open("../data/HPOs.csv", "r") as f:
    next(f)
    for line in f:
        hp_id = line.split(';')[0]
        
        # Extraire uniquement la liste contenant des IDs HP:XXXXXXX
        match = re.search(r"\[([^\]]*'HP:\d{7}'[^\]]*)\]", line)
        if match:
            parents = re.findall(r"HP:\d{7}", match.group(0))
        else:
            parents = []
        
        hp_ids.append(hp_id)
        parents_list.append(parents)

df_hpo = pd.DataFrame({'hp_id': hp_ids, 'parents': parents_list})

G_hpo_work = nx.DiGraph()
for hp_id in hp_ids:
    G_hpo_work.add_node(hp_id)
for hp_id, parents in zip(hp_ids, parents_list):
    for parent_id in parents:
        if parent_id in G_hpo_work:
            G_hpo_work.add_edge(hp_id, parent_id)

objects_w = list(G_hpo_work.nodes())
node2id_w = {n: i for i, n in enumerate(objects_w)}

In [6]:
def compute_depths0(edge_index, num_nodes):
    """
    Calcule la profondeur de chaque nœud via BFS depuis les racines.
    edge_index : (2, E) — (enfants-parent)
    """
    children = set(edge_index[0].tolist())
    roots = [n for n in range(num_nodes) if n not in children]

    depth = torch.full((num_nodes,), -1, dtype=torch.long)
    queue = deque()

    for r in roots:
        depth[r] = 0
        queue.append(r)

    # Construire liste d'adjacence parent → enfants
    adj_list = {i: [] for i in range(num_nodes)}
    for child, parent in zip(edge_index[0].tolist(), edge_index[1].tolist()):
        adj_list[parent].append(child)

    while queue:
        node = queue.popleft()
        for child in adj_list[node]:
            if depth[child] == -1:
                depth[child] = depth[node] + 1
                queue.append(child)

    max_depth = depth[depth >= 0].max().item()
    depth[depth == -1] = max_depth

    return depth


In [7]:
def read_hpoa(path):
    with open(path, 'r') as f:
        skip = sum(1 for line in f if line.startswith('#'))
    return pd.read_csv(path, sep='\t', skiprows=skip, low_memory=False)

df_hpoa = read_hpoa('../data/phenotype_omim_orpha.hpoa')
df_hpoa['disease_name'] =df_hpoa['disease_name'].str.lower().str.strip().str.replace(r'[\s\-]+', ' ', regex=True)
df_hpoa.tail()

correspondence_exacte = build_disease_correspondence(df_hpoa)
print(f"Correspondances trouvées : {len(correspondence_exacte)}")

# Construction de deux dataframes à partir de df_hpoa
df_pivot = df_hpoa[['database_id', 'hpo_id']].drop_duplicates()
df_pivot['values']=1.
df_pivot = pd.pivot_table(data=df_pivot, values='values', index='database_id', columns='hpo_id', aggfunc='max', fill_value=0)
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()

df_orpha = df_pivot[df_pivot['database_id'].str.startswith('ORPHA:')]
df_orpha = df_orpha[df_orpha['database_id'].isin(correspondence_exacte['orpha_id'])]

df_omim = df_pivot[df_pivot['database_id'].str.startswith('OMIM:')]
df_omim = df_omim[df_omim['database_id'].isin(correspondence_exacte['omim_id'])]

Correspondances trouvées : 550


In [8]:
# Avec df_omim et df_orpha issus directement de df_hpoa
omim_to_idx = {v: i for i, v in enumerate(df_omim['database_id'].values)} 
orpha_to_idx = {v: i for i, v in enumerate(df_orpha['database_id'].values)}

ground_truth = []
for _, row in correspondence_exacte.iterrows():
    if row['omim_id'] in omim_to_idx and row['orpha_id'] in orpha_to_idx:
        i = omim_to_idx[row['omim_id']]
        j = orpha_to_idx[row['orpha_id']]
        ground_truth.append((i, j))

df_gt_set = set(ground_truth)

## Fonction de coût

### Pondération par l'*Information Content*

In [9]:
weights, _, _ = compute_information_content(df_omim, G_hpo_work)
weights = {t:1/w for t, w in weights.items()}

In [10]:
def cost_matrix_hamm(df_omim, df_orpha, weights, block_size=256):
    n = df_omim.shape[0]
    m = df_orpha.shape[0]
    C = np.zeros((n, m))

    hpo_cols = [c for c in df_omim.columns if c.startswith('HP:')]
    all_hpo = list(hpo_cols)
    weights_vector = np.array([weights.get(hp, 0.0) for hp in all_hpo])

    omim_matrix = df_omim.reindex(columns=all_hpo, fill_value=0)[all_hpo].values.astype(float)
    orpha_matrix = df_orpha.reindex(columns=all_hpo,  fill_value=0)[all_hpo].values.astype(float)
    
    for i_start in tqdm(range(0, n, block_size)):
        i_end = min(i_start + block_size, n)
        block = omim_matrix[i_start:i_end]
        diff = np.abs(block[:, None, :] - orpha_matrix[None, :, :])
        C[i_start:i_end] = (diff * weights_vector).sum(axis=2)
    # Check
    for i, j in [(0, 0), (3, 7), (9, 14)]:
        ref = np.dot(np.abs(omim_matrix[i, :] - orpha_matrix[j, :]), weights_vector)
        new = C[i, j]
        print(f"C[{i},{j}]  ref={ref:.6f}  new={new:.6f}  diff={abs(ref-new):.2e}")
    return C

In [11]:
C = cost_matrix_hamm(df_omim, df_orpha, weights)

# Sans régularisation
print("======== Sans régularisation ========")
ot_plan, ot_cost = compute_transport(C, a=None, b=None)
ranks, pairs = evaluate_transport(ot_plan, df_gt_set, C)

# Avec régularisation
print("======== Avec régularisation ========")
grid = np.linspace(0.01, 5, 100)
epsilon = grid[0] * np.mean(C)
print(epsilon)
ot_plan_reg, ot_cots_reg = compute_transport_sinkhorn(C, None, None, epsilon, 10000, 1e-4, False)
ranks_reg, pairs_reg = evaluate_transport(ot_plan_reg, df_gt_set, C)

100%|██████████| 3/3 [00:31<00:00, 10.41s/it]


C[0,0]  ref=1425808.880278  new=1425808.880278  diff=0.00e+00
C[3,7]  ref=2581460.797922  new=2581460.797922  diff=0.00e+00
C[9,14]  ref=905567.703938  new=905567.703938  diff=1.16e-10
======== Sans régularisation ========
Paires évaluées : 550
Top-1 accuracy : 0.702 (386/550)
Top-3 accuracy : 0.705 (388/550)
Top-5 accuracy : 0.705 (388/550)
 Rang moyen: 76.23
======== Avec régularisation ========
17913.730145857426
Paires évaluées : 550
Top-1 accuracy : 0.693 (381/550)
Top-3 accuracy : 0.825 (454/550)
Top-5 accuracy : 0.853 (469/550)
 Rang moyen: 16.36
